In [1]:
import sys
import os

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("DeltaMergeAssignment") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
# Load customer master data
customer_master = spark.read.csv("../data/Sample - Superstore.csv", header=True, inferSchema=True)
customer_master.show(5)
customer_master.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [4]:
from pyspark.sql.functions import col, count, when

# Check nulls in each column
customer_master.select([
    count(when(col(c).isNull(), c)).alias(c) for c in customer_master.columns
]).show()

# Remove exact duplicate rows
cleaned_master = customer_master.dropDuplicates()

print("Original rows:", customer_master.count())
print("After cleaning:", cleaned_master.count())

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+

Original 

In [5]:
# Clean column names - replace spaces with underscores
new_columns = [c.replace(" ", "_").replace("-", "_") for c in cleaned_master.columns]
cleaned_master = cleaned_master.toDF(*new_columns)

cleaned_master.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [6]:
cleaned_master.write.format("delta").mode("overwrite").save("../delta/superstore_master")

In [7]:
# Verify Delta table saved correctly
result_check = spark.read.format("delta").load("../delta/superstore_master")
result_check.show(5)
print("Total rows:", result_check.count())

+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|Row_ID|      Order_ID|Order_Date|Ship_Date|     Ship_Mode|Customer_ID|   Customer_Name|    Segment|      Country|         City|       State|Postal_Code| Region|     Product_ID|       Category|Sub_Category|        Product_Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|   188|CA-2016-157000| 7/16/2016|7/22/2016|Standard Class|   AM-10360|  Alice McCarthy|  Corporate|United States|Grand Prairie|       Texas|      75051|Central|OFF-ST-10001328|Office Supplies|     Storage|Personal

In [8]:
cleaned_master.printSchema()

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Ship_Date: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [9]:
from pyspark.sql import Row

# Pick 2 existing Row_IDs to simulate updates
existing_sample = cleaned_master.select("Row_ID").limit(2).collect()
id1 = existing_sample[0]["Row_ID"]
id2 = existing_sample[1]["Row_ID"]

# Template row to keep all columns consistent
sample_row = cleaned_master.limit(1).collect()[0].asDict()

# Simulate update 1 - change Sales and Profit
update1 = sample_row.copy()
update1["Row_ID"] = id1
update1["Sales"] = "999.99"
update1["Profit"] = 150.0

# Simulate update 2 - change Ship_Mode
update2 = sample_row.copy()
update2["Row_ID"] = id2
update2["Ship_Mode"] = "Same Day"

# Get max Row_ID to create new unique IDs for inserts
max_id = cleaned_master.agg({"Row_ID": "max"}).collect()[0][0]

# Simulate insert 1 - brand new record
insert1 = sample_row.copy()
insert1["Row_ID"] = max_id + 1
insert1["Customer_Name"] = "Ravi Kumar"
insert1["City"] = "Jaipur"
insert1["Sales"] = "500.0"

# Simulate insert 2 - brand new record
insert2 = sample_row.copy()
insert2["Row_ID"] = max_id + 2
insert2["Customer_Name"] = "Anita Desai"
insert2["City"] = "Delhi"
insert2["Sales"] = "750.0"

incremental_rows = [Row(**update1), Row(**update2), Row(**insert1), Row(**insert2)]
customer_incremental = spark.createDataFrame(incremental_rows, schema=cleaned_master.schema)
customer_incremental.show(truncate=False)

+------+--------------+----------+----------+------------+-----------+-------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+---------------------------------+------+--------+--------+-------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode   |Customer_ID|Customer_Name|Segment |Country      |City     |State   |Postal_Code|Region|Product_ID     |Category |Sub_Category|Product_Name                     |Sales |Quantity|Discount|Profit |
+------+--------------+----------+----------+------------+-----------+-------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+---------------------------------+------+--------+--------+-------+
|188   |CA-2016-152156|11/8/2016 |11/11/2016|Second Class|CG-12520   |Claire Gute  |Consumer|United States|Henderson|Kentucky|42420      |South |FUR-BO-10001798|Furniture|Bookcases   |Bush Somerset Collection Bookcase|999.99|2       |0       |1

In [10]:
from delta.tables import DeltaTable

# Load the existing Delta table as a DeltaTable object (needed for MERGE)
delta_table = DeltaTable.forPath(spark, "../delta/superstore_master")

# Perform the MERGE (upsert)
delta_table.alias("target").merge(
    customer_incremental.alias("source"),
    "target.Row_ID = source.Row_ID"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("MERGE completed successfully!")

MERGE completed successfully!


In [11]:
# Reload the Delta table after merge
result = spark.read.format("delta").load("../delta/superstore_master")

print("Total row count after merge:", result.count())

# Check for duplicate Row_IDs (should be none)
result.groupBy("Row_ID").count().filter("count > 1").show()

# Verify the 2 updated records
result.filter(result.Row_ID.isin([existing_sample[0]["Row_ID"], existing_sample[1]["Row_ID"]])).show(truncate=False)

# Verify the 2 new inserted records
result.filter(result.Row_ID.isin([max_id + 1, max_id + 2])).show(truncate=False)

# Check Delta transaction history (shows MERGE operation logged)
delta_table.history().show(truncate=False)

Total row count after merge: 9996
+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+

+------+--------------+----------+----------+------------+-----------+-------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+---------------------------------+------+--------+--------+-------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode   |Customer_ID|Customer_Name|Segment |Country      |City     |State   |Postal_Code|Region|Product_ID     |Category |Sub_Category|Product_Name                     |Sales |Quantity|Discount|Profit |
+------+--------------+----------+----------+------------+-----------+-------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+---------------------------------+------+--------+--------+-------+
|188   |CA-2016-152156|11/8/2016 |11/11/2016|Second Class|CG-12520   |Claire Gute  |Consumer|United States|Henderson|Kentucky|42420      |South |FUR-

In [12]:
# Final display of complete dataset
result.orderBy("Row_ID").show(20, truncate=False)

# Summary
original_count = customer_master.count()
final_count = result.count()
updates_count = 2
inserts_count = 2

print("=" * 50)
print("MERGE OPERATION SUMMARY")
print("=" * 50)
print(f"Original row count       : {original_count}")
print(f"Final row count          : {final_count}")
print(f"Records updated          : {updates_count}")
print(f"Records inserted         : {inserts_count}")
print(f"Duplicate Row_IDs found  : 0")
print("=" * 50)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------------------------+--------+--------+--------+--------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name     |Segment    |Country      |City           |State         |Postal_Code|Region |Product_ID     |Category       |Sub_Category|Product_Name                                                                |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152

## Summary

- **Objective**: Perform incremental data processing on the Superstore dataset using Delta Lake MERGE.
- **Steps**: Loaded raw CSV → cleaned nulls/duplicates → saved as Delta table → created incremental dataset (2 updates + 2 inserts) → applied MERGE using Row_ID as key → validated results.
- **Result**: MERGE correctly updated 2 existing records and inserted 2 new records, with no duplicate Row_IDs after the operation.
- **Learning**: Delta Lake's MERGE (upsert) allows updating and inserting in a single atomic operation, unlike traditional overwrite-based approaches.